# Task 4 — Valutazione dei classificatori manuali su `training.csv`

> 📄 Documentazione completa e motivazioni: [`docs/task4.md`](../docs/task4.md)

L'obiettivo e` quello di valutare i classificatori implementati nel Task 2 su un dataset piu ampio , cercando di **ottimizzarne le prestazioni**.

## 0. Setup 

In questo primo step , vengono importate le librerie e il dataset `training.csv`.


In [42]:
import pandas as pd
import numpy as np


In [43]:
training = pd.read_csv("../data/processed/training.csv", sep=";")
dataFrame_NB = training.copy() 

## 1. Richiamo di variabili e metodi utilizzati in Naive Bayes
In questa sezione riprendiamo variabili e metodi necessati per testare Naive Bayes su `training.csv`.
In particolare andremo a riprendere **predici_naive_bayes** ,che implementa il modello vero e proprio, e **probabilita**. 



In [44]:
#Tutte le probabilità condizionate calcolate con lo stimatore di Laplace.

probabilita = {
    "marital": {
        0: {"divorced": 1/9, "married": 7/9, "single": 1/9},
        1: {"divorced": 1/6, "married": 2/6, "single": 3/6},
    },
    "housing": {
        0: {"no": 2/6, "yes": 4/6},
        1: {"no": 2/6, "yes": 4/6},
    },
    "loan": {
        0: {"no": 5/6, "yes": 1/6},
        1: {"no": 7/8, "yes": 1/8},
    }
}

In [45]:
def predici_naive_bayes(row):
    # Probabilità a priori calcolate nel task 2
    score_0 = 0.5
    score_1 = 0.5

    # marital
    score_0 *= probabilita["marital"][0][row["marital"]] 
    score_1 *= probabilita["marital"][1][row["marital"]]

    # housing
    score_0 *= probabilita["housing"][0][row["housing"]]
    score_1 *= probabilita["housing"][1][row["housing"]]

    # loan
    score_0 *= probabilita["loan"][0][row["loan"]]
    score_1 *= probabilita["loan"][1][row["loan"]]



    # Predizione finale 
    if (score_0 > score_1):
        prediction = 0
    else:
        prediction = 1
    return prediction, score_0, score_1

## 2. Naïve Bayes su `training.csv`

Una volta ripreso cio che ci serve , possiamo testare il modello su `training.csv`.

In [46]:
dataFrame_NB["Predicted"] = dataFrame_NB.apply(lambda row: predici_naive_bayes(row)[0],axis=1)
dataFrame_NB[["y", "Predicted"]].head(10)

,y,Predicted
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,1
9,0,1


## 3. Valutazione delle prestazioni 

Una volta applicato il modello sul dataset `training.csv`, possiamo valutare le prestazioni basandoci sui seguenti criteri :

- Confusion Matrix
- Accuracy
- Precision
- Recall
- F1

I parametri utilizzati per calcolarci tali metriche di valutazione sono :

- TP = True Positive
- TN = True Negative
- FP = False Positive
- FN = False Negative



## 3.1 Confusion Matrix

In [47]:

tp = len(dataFrame_NB[(dataFrame_NB["y"] == 1) & (dataFrame_NB["Predicted"] == 1)])
tn = len(dataFrame_NB[(dataFrame_NB["y"] == 0) & (dataFrame_NB["Predicted"] == 0)])
fp = len(dataFrame_NB[(dataFrame_NB["y"] == 0) & (dataFrame_NB["Predicted"] == 1)])
fn = len(dataFrame_NB[(dataFrame_NB["y"] == 1) & (dataFrame_NB["Predicted"] == 0)])


confusion_matrix_df = pd.DataFrame(
    [[tn, fp],
     [fn, tp]],
    columns=["Predetto 0", "Predetto 1"],
    index=["Reale 0", "Reale 1"]
)
print("TP = ", tp)
print("TN = ", tn)
print("FP = ", fp)
print("FN = ", fn)
print("\nMatrice di Confusione:")
confusion_matrix_df



TP =  2096
TN =  22458
FP =  14079
FN =  2543

Matrice di Confusione:


,Predetto 0,Predetto 1
Reale 0,22458,14079
Reale 1,2543,2096


## 3.2 Accuracy

L'**accuracy** indica quanto e stato bravo il modello.

In [48]:
accuracy = (dataFrame_NB["y"] == dataFrame_NB["Predicted"]).mean()
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5963


## 3.3 Precision

La **precision** indica in percentuale quanti tra tutti i positivi individuati sono realmente positivi.

$$ \text{Precision} = \frac{TP}{TP + FP} $$

In [49]:
precision = tp / (tp + fp)
print(f"Precision: {precision:.4f}")

Precision: 0.1296


## 3.4 Recall

La **recall** indica in percentuale quanti positivi sono stati individuati

$$ \text{Recall} = \frac{TP}{TP + FN} $$

In [50]:
recall = tp / (tp + fn)
print(f"Recall: {recall:.4f}")

Recall: 0.4518


## 3.5 F1

L' **F1** rappresenta la media ponderata tra precision e recall.

$$ F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} $$

In [51]:
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
print(f"F1 Score: {f1:.4f}")

F1 Score: 0.2014


## 4. Analisi dei risultati

Il classificatore Naive Bayes ha ottenuto i seguenti risultati :

| Metrica | Valore |
|---|---|
| Accuracy  | 59.63% |
| Precision | 12.96% |
| Recall    | 45.18% |
| F1-Score  | 20.14% |

con la seguente matrice di confusione:

|            | Predicted 0 | Predicted 1 |
|------------|:-----------:|:-----------:|
| **Actual 0** | 22458 | 14079 |
| **Actual 1** | 2543 | 2096 |

L'accuracy risulta moderata, ma non rappresenta da sola una misura affidabile delle prestazioni del classificatore, poiché il dataset presenta un forte sbilanciamento tra le classi.

La precision è molto bassa (12.96%), indicando che il classificatore tende a produrre molti falsi positivi. In altre parole, numerosi clienti vengono classificati come non propensi a stipulare un contratto con la banca.

La recall risulta inferiore al 50% (45.18%), cio mosta che il modello individua discretamente i reali positivi.

L'F1-score, pari al 20.14%, evidenzia  che il compromesso tra precision e recall  è  tutt`altro che ottimale.



### 5. Ottimizzazione

Avendo scelto di valutare il modello senza riaddestrarlo , il margine di
ottimizzazione disponibile in questo task è **limitato**: 

Qui abbiamo solo cambiato le probabilita a priori basandoci sui risultati ottenuti nel task 1.

Le leve di ottimizzazione vere e proprie come ad esempio  **riaddestrare** i modelli su `training.csv`,
**aggiungere attributi** più informativi, **discretizzare** le variabili numeriche, e soprattutto
**gestire lo sbilanciamento** delle classi (es. con pesi di classe o ricampionamento), 
 vengono trattate nel**Task 5**.


In [52]:
def predici_naive_bayes_ottimizzato(row):
    # Probabilità a priori cambiate in base alla distribuzione reale del target nel training set
    score_0 = 0.887
    score_1 = 0.113

    # marital
    score_0 *= probabilita["marital"][0][row["marital"]] 
    score_1 *= probabilita["marital"][1][row["marital"]]

    # housing
    score_0 *= probabilita["housing"][0][row["housing"]]
    score_1 *= probabilita["housing"][1][row["housing"]]

    # loan
    score_0 *= probabilita["loan"][0][row["loan"]]
    score_1 *= probabilita["loan"][1][row["loan"]]

    # Predizione finale 
    if (score_0 > score_1):
        prediction = 0
    else:
        prediction = 1
    return prediction, score_0, score_1

In [53]:
dataFrame_NB["Predicted"] = dataFrame_NB.apply(lambda row: predici_naive_bayes_ottimizzato(row)[0],axis=1)
dataFrame_NB[["y", "Predicted"]].head(10)

,y,Predicted
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


## 6. Valutazione prestazioni modello ottimizzato

Di seguito verranno ricalcolate tutte le metriche basandosi sulle nuove probabilita a priori

In [54]:
tp_opt = len(dataFrame_NB[(dataFrame_NB["y"] == 1) & (dataFrame_NB["Predicted"] == 1)])
tn_opt = len(dataFrame_NB[(dataFrame_NB["y"] == 0) & (dataFrame_NB["Predicted"] == 0)])
fp_opt = len(dataFrame_NB[(dataFrame_NB["y"] == 0) & (dataFrame_NB["Predicted"] == 1)])
fn_opt = len(dataFrame_NB[(dataFrame_NB["y"] == 1) & (dataFrame_NB["Predicted"] == 0)])


confusion_matrix_df = pd.DataFrame(
    [[tn_opt, fp_opt],
     [fn_opt, tp_opt]],
    columns=["Predetto 0", "Predetto 1"],
    index=["Reale 0", "Reale 1"]
)
accuracy = (dataFrame_NB["y"] == dataFrame_NB["Predicted"]).mean()
precision = tp_opt / (tp_opt + fp_opt) if( tp_opt + fp_opt ) > 0 else 0.0
recall = tp_opt / (tp_opt + fn_opt) if (tp_opt + fn_opt) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if  (precision + recall) > 0 else 0.0

print("TP = ", tp_opt)
print("TN = ", tn_opt)
print("FP = ", fp_opt)
print("FN = ", fn_opt)
print("Accuracy = ", accuracy)
print("Precision = ", precision)
print("Recall = ", recall)
print("F1 Score = ", f1_score)
print("\nMatrice di Confusione:")
confusion_matrix_df

TP =  0
TN =  36537
FP =  0
FN =  4639
Accuracy =  0.8873372838546726
Precision =  0.0
Recall =  0.0
F1 Score =  0.0

Matrice di Confusione:


,Predetto 0,Predetto 1
Reale 0,36537,0
Reale 1,4639,0


## 7. Confronto tra modello originale e modello ottimizzato

Dopo aver applicato il classificatore Naive Bayes manuale al dataset training.csv, sono state calcolate le metriche di performance sia per il modello originale (basato sulle probabilità stimate da manuale.csv) sia per il modello ottimizzato (con probabilità a priori ricalcolate sull intero set).

| Metrica | Modello Originale | Modello Ottimizzato |
|---|---|---|
| Accuracy  | 59.63% | 88.73%|
| Precision | 12.96% | 0%|
| Recall    | 45.18% | 0%|
| F1-Score  | 20.14% | 0%|


#### **Considerazioni**